# 04 Module 4 federation and final evaluation

Run these notebooks in order, one cell at a time. Inspect each result before continuing.
Use a fresh Colab runtime for the pinned dependencies. CPU is sufficient for this small ANN;
GPU is optional. Use the same RUN_ID in all notebooks. Existing results are never overwritten.
The notebooks import repository code, so training and deployment use one implementation.
No model quality result is prefilled or claimed here.

In [ ]:
import base64, os, sys, subprocess
from pathlib import Path
REPO = Path('/content/FedHealth-four-class')
BRANCH = 'feat/four-class-ml-federation'
if not REPO.exists():
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception as exc:
        raise RuntimeError('Add GITHUB_TOKEN in Colab Secrets and enable notebook access.') from exc
    if not token:
        raise RuntimeError('GITHUB_TOKEN is empty in Colab Secrets.')
    credential = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['GIT_CONFIG_COUNT'] = '1'
    env['GIT_CONFIG_KEY_0'] = 'http.https://github.com/.extraheader'
    env['GIT_CONFIG_VALUE_0'] = f'AUTHORIZATION: basic {credential}'
    result = subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch',
         'https://github.com/ahsen-dev86/FedHealth-FYP.git', str(REPO)],
        env=env, text=True, capture_output=True,
    )
    if result.returncode:
        raise RuntimeError(f'Git clone failed:\n{result.stderr.strip()}')
else:
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'{REPO} exists but is not a Git checkout')
    branch = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
    if branch != BRANCH:
        raise RuntimeError(f'Use a fresh runtime: existing checkout is {branch}')
print('Branch:', BRANCH)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
assert (3, 11) <= sys.version_info[:2] <= (3, 13), 'Use Python 3.11–3.13 (verified locally on 3.12)'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'Module 4 FL/requirements.txt')])
print('Dependencies installed. Restart the Colab session after the first install, then continue.')

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
REPO = Path('/content/FedHealth-four-class')
from google.colab import drive
drive.mount('/content/drive')
# Keep this identical across all four notebooks. Use a new ID for a new experiment.
RUN_ID = 'four_class_v1'
RUN = Path('/content/drive/MyDrive/FYP/four_class') / RUN_ID
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'Module 4 FL'))
from fedhealth_ml.data import Dataset
from fedhealth_ml.training import DEFAULT_CSV, DEFAULT_PROVENANCE
from fedhealth_ml.runtime import write_json
dataset = Dataset(DEFAULT_CSV, DEFAULT_PROVENANCE)
print('Run:', RUN)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Dataset:', dataset.sha256)

Module 4 fine-tunes the exact saved ANN. Only ANN weights, counts and metrics are exchanged. The Colab orchestrator simulates public-data labs; it is not a demonstration of patient-data privacy. XGBoost is refit centrally as a reference after ANN selection and is never averaged.

In [ ]:
from run_simulation import run
smoke = run(dataset, RUN / 'seed42/centralized/model', RUN / 'smoke',
            rounds=2, labs=3, local_epochs=1, seed=42)
print('Smoke complete:', smoke)
display(json.loads((RUN / 'smoke/round_history.json').read_text())['aggregation'])

Check that all 3 labs contributed in both rounds and the total is 2,240 rows. Local Adam is reset every round; rate 0.0001, batch 32, one local epoch. Round 0 is eligible and may win. The smoke is excluded from final comparisons.

In [ ]:
from run_experiments import suite
# Fixed stability protocol: three seeds, four partitions, 20 rounds each.
# Reuses verified seed 42 central artifacts; trains fresh central models for 52 and 62.
comparison = suite(dataset, RUN, seeds=(42, 52, 62), rounds=20, epochs=100)
import pandas as pd
display(pd.DataFrame([{'seed': r['seed'], 'experiment': r['experiment'],
                      'ann_f1': r['metrics']['ann']['macro_f1'],
                      'hybrid_f1': r['metrics']['hybrid']['macro_f1']}
                     for r in comparison]))

Every round is validated. The selected round has the lowest validation log loss among rounds with macro-F1 no more than 0.05 below round 0 and no extra sick-to-healthy errors. These are experiment selection rules, not clinical safety guarantees. Report every seed and every partition; do not select the best seed.

In [ ]:
# Run only after reviewing validation and freezing all settings.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    from fedhealth_ml.final_evaluate import final_evaluate
    final = final_evaluate(dataset, RUN / 'final_evaluation_plan.json', RUN / 'final_test')
    print('Final test saved:', RUN / 'final_test')
else:
    print('Final test remains locked. Set RUN_FINAL_TEST=True only after protocol freeze.')

Run the following sensitivity check after the final comparison. It trains fresh models and preprocessors excluding each source. Do not tune the main models from its results. Upstream donor imputation still limits source independence.

In [ ]:
RUN_SOURCE_HOLDOUT = False
if RUN_SOURCE_HOLDOUT:
    from fedhealth_ml.source_holdout import run as source_check
    source_check(dataset, RUN / 'source_holdout', seed=42)
else:
    print('Source sensitivity not yet run.')

Final deliverables: model bundles and hashes, validation round histories, all-seed comparison, final class/subtype/source recall and ECE, source sensitivity, and negative findings. Module 5 encrypted aggregation and full Module 9 deployment are separate verification tasks.